<a href="https://colab.research.google.com/github/ananthchavancs24-prog/AI/blob/main/Lab_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# FOL Forward Chaining Algorithm

# Initialize Knowledge Base with facts
KB = {
    ("American", "Robert"),
    ("Missile", "T1"),
    ("Owns", "A", "T1"),
    ("Enemy", "A", "America")
}

# Initialize rules
rules = [
    # If x is a missile and A owns x, Robert sells x to A
    (["Missile(x)", "Owns(A,x)"], "Sells(Robert,x,A)"),

    # If x is a missile, x is a weapon
    (["Missile(x)"], "Weapon(x)"),

    # If x is an enemy of America, x is hostile
    (["Enemy(x,America)"], "Hostile(x)"),

    # If all conditions are satisfied, p is criminal
    (["American(p)", "Weapon(q)",
      "Sells(p,q,r)", "Hostile(r)"], "Criminal(p)")
]


# Unification function
def match(pattern, fact, substitution):
    pattern = pattern.replace("(", " ").replace(")", "")
    pattern = pattern.split(",") if "," in pattern else pattern.split()

    # Extract predicate and arguments
    fact_predicate = fact[0]
    fact_args = fact[1:]

    pattern_predicate = pattern[0]
    pattern_args = [x.strip() for x in pattern[1:]]

    if pattern_predicate != fact_predicate:
        return None

    if len(pattern_args) != len(fact_args):
        return None

    sub = substitution.copy()

    for p, f in zip(pattern_args, fact_args):
        if p.islower():
            if p in sub and sub[p] != f:
                return None
            sub[p] = f
        elif p != f:
            return None

    return sub


# Apply substitution to conclusion
def substitute(conclusion, sub):
    for var, value in sub.items():
        conclusion = conclusion.replace(var, value)
    return conclusion


# Forward chaining algorithm
def forward_chaining(KB, rules):

    while True:
        new = set()

        for premises, conclusion in rules:

            substitutions = [{}]

            for premise in premises:
                updated = []

                for sub in substitutions:
                    for fact in KB:
                        result = match(premise, fact, sub)

                        if result is not None:
                            updated.append(result)

                substitutions = updated

            for sub in substitutions:
                inferred = substitute(conclusion, sub)

                # Convert inferred fact into tuple
                parts = inferred.replace("(", " ").replace(")", "")
                parts = parts.replace(",", " ").split()

                new_fact = tuple(parts)

                if new_fact not in KB:
                    new.add(new_fact)

        if not new:
            break

        print("New facts inferred:", new)

        KB.update(new)

    return KB


# Execute forward chaining
KB = forward_chaining(KB, rules)

# Check the query
query = ("Criminal", "Robert")

if query in KB:
    print("\nRobert is Criminal")
else:
    print("\nRobert is not proved Criminal")

New facts inferred: {('Weapon', 'T1')}

Robert is not proved Criminal
